In [ ]:
!pip install torch torchvision matplotlib opencv-python


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

# Setup CUDA GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20

DATASET_PATH = "/kaggle/input/anemia-detection/Anemia using eyes"


In [ ]:
# Data Augmentation & Loading
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, scale=(0.9, 1.1)), # Equivalent to zoom_range
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
class_names = full_dataset.classes
print(f'Classes: {class_names}')

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}')


In [ ]:
# Display sample images
images, labels = next(iter(train_loader))
plt.figure(figsize=(8,8))
for i in range(min(6, len(images))):
    plt.subplot(2,3,i+1)
    # PyTorch uses (C, H, W). Matplotlib needs (H, W, C).
    img = images[i].permute(1, 2, 0).numpy()
    plt.imshow(img)
    plt.title("Anemic" if labels[i].item() == 0 else "Non-Anemic")
    plt.axis("off")
plt.show()


In [ ]:
# Define CNN Model
class AnemiaCNN(nn.Module):
    def __init__(self):
        super(AnemiaCNN, self).__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=0)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Conv Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=0)
        
        # Conv Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=0)
        
        # Calculate flattened size based on input 224x224 and valid padding (0)
        self.flattened_size = 128 * 26 * 26
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(self.flattened_size, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 1)
        
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(-1, self.flattened_size)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.fc2(x)) # Sigmoid for binary classification
        return x

model = AnemiaCNN().to(device)
print(model)


In [ ]:
# Loss and Optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())


In [ ]:
# Training Loop
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        # Move to GPU
        images = images.to(device)
        labels = labels.to(device).float().view(-1, 1)
        
        # Forward pass & backprop
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        # Metrics
        running_loss += loss.item() * images.size(0)
        predicted = (outputs > 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_losses.append(running_loss / total)
    train_accs.append(correct / total)
    
    # Validation phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device).float().view(-1, 1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            predicted = (outputs > 0.5).float()
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_losses.append(val_loss / val_total)
    val_accs.append(val_correct / val_total)
    
    print(f'Epoch [{epoch+1}/{EPOCHS}] | '
          f'Train Loss: {train_losses[-1]:.4f}, Acc: {train_accs[-1]:.4f} | '
          f'Val Loss: {val_losses[-1]:.4f}, Val Acc: {val_accs[-1]:.4f}')


In [ ]:
plt.figure(figsize=(12,5))

# Accuracy Plot
plt.subplot(1,2,1)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Model Accuracy')

# Loss Plot
plt.subplot(1,2,2)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Model Loss')

plt.show()


In [ ]:
# Save the weights to a PyTorch file (.pth)
torch.save(model.state_dict(), "anemia_eye_cnn_model.pth")
print("Model saved successfully!")
